#LangChain + RAG Agent - Lo mejor de ambos mundos
   
   Este código implementa un agente inteligente que combina:
   1. LangChain Agent (para razonamiento y uso de herramientas)
   2. RAG (Retrieval-Augmented Generation) para búsqueda semántica
   3. Pandas (para cálculos estadísticos)
   4. Mistral AI (como modelo de lenguaje)

# CELDA 1 - INSTALACIÓN DE DEPENDENCIAS


In [1]:
# -*- coding: utf-8 -*-
# CELDA 1 - Instalación de la ÚLTIMA VERSIÓN de LangChain

# Primero, desinstalar versiones viejas
!pip uninstall -y langchain langchain-community langchain-core langchain-experimental langchain-mistralai chromadb

# Instalar las últimas versiones
!pip install --upgrade langchain==0.3.0
!pip install --upgrade langchain-community==0.3.0
!pip install --upgrade langchain-core==0.3.0
!pip install --upgrade langchain-experimental==0.3.0
!pip install --upgrade langchain-mistralai==0.2.0
!pip install --upgrade chromadb==0.5.0
!pip install --upgrade sentence-transformers
!pip install pandas openpyxl thefuzz python-Levenshtein

print('✅ Últimas versiones instaladas')

# Verificar versiones
print('\n📦 Versiones instaladas:')
!pip show langchain | grep Version
!pip show langchain-community | grep Version
!pip show langchain-mistralai | grep Version

Found existing installation: langchain 0.3.0
Uninstalling langchain-0.3.0:
  Successfully uninstalled langchain-0.3.0
Found existing installation: langchain-core 0.3.63
Uninstalling langchain-core-0.3.63:
  Successfully uninstalled langchain-core-0.3.63
  Using cached langchain-0.3.0-py3-none-any.whl.metadata (7.1 kB)
  Using cached langchain_core-0.3.86-py3-none-any.whl.metadata (3.2 kB)
INFO: pip is looking at multiple versions of langchain-core to determine which version is compatible with other requirements. This could take a while.
  Using cached langchain_core-0.3.85-py3-none-any.whl.metadata (3.2 kB)
  Using cached langchain_core-0.3.84-py3-none-any.whl.metadata (3.2 kB)
  Using cached langchain_core-0.3.83-py3-none-any.whl.metadata (3.2 kB)
  Using cached langchain_core-0.3.82-py3-none-any.whl.metadata (3.2 kB)
  Using cached langchain_core-0.3.81-py3-none-any.whl.metadata (3.2 kB)
  Using cached langchain_core-0.3.80-py3-none-any.whl.metadata (3.2 kB)
  Using cached langchain_

🔧 CELDA 2 - Importaciones para LangChain 0.3.x

In [3]:
# -*- coding: utf-8 -*-
# CELDA 2 - IMPORTACIONES para LangChain 0.3.x

import pandas as pd
import numpy as np
import re
import unicodedata
from thefuzz import fuzz, process
from google.colab import files, userdata

# LangChain Core - Estos funcionan en 0.3.x
from langchain_core.documents import Document
from langchain_core.callbacks import StdOutCallbackHandler
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# LangChain Community - Vector stores y herramientas
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import DataFrameLoader

# LangChain Mistral AI
from langchain_mistralai import ChatMistralAI, MistralAIEmbeddings

# LangChain Agents (nueva forma en 0.3.x)
from langchain.agents import AgentExecutor
from langchain.tools import tool
from langchain.memory import ConversationBufferMemory

# Para el agente con herramientas
from langchain.agents import create_tool_calling_agent
from langchain.agents import AgentExecutor

import warnings
warnings.filterwarnings('ignore')

print('✅ LangChain 0.3.x cargado correctamente')
print(f'📚 Versión LangChain: {pd.__version__}')

✅ LangChain 0.3.x cargado correctamente
📚 Versión LangChain: 2.2.2


# CELDA 3 - FUNCIONES DE NORMALIZACIÓN DE DATOS

In [4]:
# -*- coding: utf-8 -*-
# CELDA 3 - Funciones de normalización

def quitar_tildes(texto):
    """Elimina tildes y diacríticos"""
    texto = unicodedata.normalize('NFKD', str(texto))
    return texto.encode('ascii', 'ignore').decode('ascii')

def limpiar_texto(texto):
    """Limpieza suave para nombres propios"""
    if pd.isna(texto):
        return np.nan
    texto = str(texto)
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

def normalizar_texto_lower(texto):
    """Limpieza completa para categóricas"""
    if pd.isna(texto):
        return np.nan
    texto = quitar_tildes(str(texto))
    texto = texto.lower()
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

def normalizar_nombre_columna(nombre):
    """Convierte a snake_case"""
    nombre = quitar_tildes(str(nombre))
    nombre = nombre.lower()
    nombre = re.sub(r'[\s\-]+', '_', nombre)
    nombre = re.sub(r'[^\w]', '', nombre)
    return nombre

def normalizar_fecha(fecha):
    """Convierte a datetime"""
    if pd.isna(fecha):
        return np.nan
    try:
        return pd.to_datetime(fecha, format='%m/%d/%Y %H:%M', errors='coerce')
    except:
        return pd.to_datetime(fecha, errors='coerce')

def normalizar_telefono(telefono):
    """Estandariza números de teléfono"""
    if pd.isna(telefono):
        return np.nan
    telefono = re.sub(r'[^+\d]', '', str(telefono))
    return telefono if telefono else np.nan

print('✅ Funciones de normalización cargadas')

✅ Funciones de normalización cargadas


# CELDA 4 - CARGA DEL ARCHIVO CSV


In [6]:
# -*- coding: utf-8 -*-
# CELDA 4 - Cargar y limpiar datos

# Subir archivo
uploaded = files.upload()
nombre_archivo = list(uploaded.keys())[0]

# Cargar CSV
try:
    df = pd.read_csv(nombre_archivo, encoding='utf-8')
except UnicodeDecodeError:
    df = pd.read_csv(nombre_archivo, encoding='latin-1')
    print('⚠ Archivo leído con latin-1')

print(f'✅ Archivo cargado: {df.shape[0]} filas, {df.shape[1]} columnas')

# Normalizar columnas
df.columns = [normalizar_nombre_columna(col) for col in df.columns]

# Eliminar filas vacías y duplicados
df.dropna(how='all', inplace=True)
df.drop_duplicates(inplace=True)

# Procesar fechas
if 'orderdate' in df.columns:
    df['orderdate'] = df['orderdate'].apply(normalizar_fecha)
    df['order_year_month'] = df['orderdate'].dt.strftime('%Y-%m')
    df['year_id'] = df['orderdate'].dt.year

# Normalizar teléfonos
if 'phone' in df.columns:
    df['phone'] = df['phone'].apply(normalizar_telefono)

# Limpiar texto en columnas específicas
columnas_nombre = ['customername', 'contactlastname', 'contactfirstname',
                   'addressline1', 'city', 'state', 'country']
for col in columnas_nombre:
    if col in df.columns:
        df[col] = df[col].apply(limpiar_texto)

# Normalizar categóricas
columnas_categoricas = ['productline', 'dealsize', 'status', 'territory']
for col in columnas_categoricas:
    if col in df.columns:
        df[col] = df[col].apply(normalizar_texto_lower)

print('✅ Limpieza completada')

# Manejo de nulos
for col in ['state', 'postalcode', 'territory']:
    if col in df.columns:
        df[col] = df[col].fillna('unknown')

df.dropna(subset=['customername', 'country', 'sales'], inplace=True)

print(f'✅ Dataset final: {df.shape[0]} filas, {df.shape[1]} columnas')

Saving sales_data_sample.csv to sales_data_sample.csv
⚠ Archivo leído con latin-1
✅ Archivo cargado: 2823 filas, 25 columnas
✅ Limpieza completada
✅ Dataset final: 2823 filas, 26 columnas


🔍 CELDA 5 - Fuzzy matching (opcional pero útil)

In [7]:
# -*- coding: utf-8 -*-
# CELDA 5 - Unificar clientes similares

UMBRAL_SIMILITUD = 85

def deduplicar_columna(serie, umbral):
    """Agrupa valores similares bajo el más frecuente"""
    valores_unicos = serie.dropna().unique().tolist()
    mapa = {}
    procesados = set()

    for valor in valores_unicos:
        if valor in procesados:
            continue

        similares = process.extract(
            valor, valores_unicos,
            scorer=fuzz.token_sort_ratio,
            limit=50
        )

        grupo = [v for v, score in similares if score >= umbral]
        frecuencias = serie[serie.isin(grupo)].value_counts()
        canonico = frecuencias.idxmax()

        for v in grupo:
            mapa[v] = canonico
            procesados.add(v)

    return serie.map(lambda x: mapa.get(x, x))

if 'customername' in df.columns:
    antes = df['customername'].nunique()
    df['customername'] = deduplicar_columna(df['customername'], UMBRAL_SIMILITUD)
    despues = df['customername'].nunique()
    print(f'✅ Clientes unificados: {antes} → {despues}')

✅ Clientes unificados: 92 → 91


🎯 CELDA 6 - Feature engineering

In [8]:
# -*- coding: utf-8 -*-
# CELDA 6 - Crear nuevas columnas

# Flag para deals grandes
if 'dealsize' in df.columns:
    df['is_large_deal'] = (df['dealsize'] == 'large').astype(int)

# Margen de ganancia
if all(col in df.columns for col in ['sales', 'quantityordered', 'msrp']):
    df['profit_margin'] = np.where(
        df['sales'] > 0,
        (df['sales'] - df['quantityordered'] * df['msrp']) / df['sales'],
        0
    ).round(4)

print('✅ Feature engineering completado')
print(f'Columnas disponibles: {", ".join(df.columns.tolist())}')

✅ Feature engineering completado
Columnas disponibles: ordernumber, quantityordered, priceeach, orderlinenumber, sales, orderdate, status, qtr_id, month_id, year_id, productline, msrp, productcode, customername, phone, addressline1, addressline2, city, state, postalcode, country, territory, contactlastname, contactfirstname, dealsize, order_year_month, is_large_deal, profit_margin


🔑 CELDA 7 - Configurar API Key

In [9]:
# -*- coding: utf-8 -*-
# CELDA 7 - API Key de Mistral

try:
    MISTRAL_API_KEY = userdata.get('MISTRAL_API_KEY')
    if not MISTRAL_API_KEY:
        raise ValueError("Secret vacío")
    print('✅ API key cargada desde Colab Secrets')
except:
    MISTRAL_API_KEY = input('Pegá tu MISTRAL_API_KEY: ').strip()
    print('✅ API key configurada manualmente')

✅ API key cargada desde Colab Secrets


🚀 CELDA 8 - Construir Vector Store (RAG)


In [10]:
# -*- coding: utf-8 -*-
# CELDA 8 - RAG con ChromaDB

from langchain_core.documents import Document

def crear_documentos(df, limite=5000):
    """Convierte DataFrame en documentos para RAG"""
    documentos = []

    # Tomar muestra si es muy grande
    if len(df) > limite:
        df_uso = df.sample(n=limite, random_state=42)
        print(f"📊 Usando muestra de {limite} filas (de {len(df)} totales)")
    else:
        df_uso = df
        print(f"📊 Usando todas las {len(df_uso)} filas")

    for idx, row in df_uso.iterrows():
        # Texto rico para búsqueda semántica
        texto = f"""
        Venta ID: {row.get('ordernumber', 'N/A')}
        Cliente: {row.get('customername', 'N/A')}
        País: {row.get('country', 'N/A')}
        Ciudad: {row.get('city', 'N/A')}
        Producto: {row.get('productline', 'N/A')}
        Cantidad: {row.get('quantityordered', 0)}
        Venta Total: ${row.get('sales', 0):,.2f}
        Estado: {row.get('status', 'N/A')}
        Año: {row.get('year_id', 'N/A')}
        """

        metadata = {
            'customername': str(row.get('customername', '')),
            'country': str(row.get('country', '')),
            'productline': str(row.get('productline', '')),
            'year': str(row.get('year_id', ''))
        }

        documentos.append(Document(page_content=texto, metadata=metadata))

    return documentos

# Crear documentos
print('🔨 Creando documentos para RAG...')
documentos = crear_documentos(df, limite=5000)

# Configurar embeddings y vector store
print('🔨 Generando embeddings...')
embeddings = MistralAIEmbeddings(
    api_key=MISTRAL_API_KEY,
    model="mistral-embed"
)

print('🔨 Construyendo vector store...')
vectorstore = Chroma.from_documents(
    documents=documentos,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

# Configurar retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 10}
)

print(f'✅ Vector store listo con {len(documentos)} documentos')

🔨 Creando documentos para RAG...
📊 Usando todas las 2823 filas
🔨 Generando embeddings...


tokenizer.json: 0.00B [00:00, ?B/s]

🔨 Construyendo vector store...


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


✅ Vector store listo con 2823 documentos


🛠️ CELDA 9 - Crear herramientas para el agente

In [11]:
# -*- coding: utf-8 -*-
# CELDA 9 - Herramientas del agente

from langchain.tools import tool

@tool
def buscar_rag(pregunta: str) -> str:
    """
    Busca información específica en el dataset usando búsqueda semántica.
    Útil para preguntas sobre clientes, productos, países específicos.
    """
    docs = retriever.invoke(pregunta)

    if not docs:
        return "No se encontró información relevante."

    resultado = f"🔍 Encontradas {len(docs)} transacciones relevantes:\n\n"
    for i, doc in enumerate(docs[:5], 1):
        resultado += f"--- Resultado {i} ---\n{doc.page_content}\n\n"

    return resultado

@tool
def calcular_pandas(operacion: str) -> str:
    """
    Ejecuta cálculos con pandas. Pasar operación como string.
    Ejemplos: "df['sales'].sum()", "df.groupby('country')['sales'].mean()"
    """
    try:
        resultado = eval(operacion)
        if isinstance(resultado, (pd.Series, pd.DataFrame)):
            return str(resultado.head(20))
        elif isinstance(resultado, float):
            return f"{resultado:,.2f}"
        return str(resultado)
    except Exception as e:
        return f"Error: {str(e)}"

@tool
def resumen_dataset() -> str:
    """Devuelve resumen estadístico del dataset"""
    return f"""
    📊 RESULTADOS:
    - Filas: {len(df):,}
    - Columnas: {len(df.columns)}
    - Ventas totales: ${df['sales'].sum():,.2f}
    - Países: {df['country'].nunique()}
    - Clientes: {df['customername'].nunique()}
    - Años: {sorted(df['year_id'].dropna().unique()) if 'year_id' in df.columns else 'N/A'}
    """

print('✅ Herramientas creadas')
print('   - buscar_rag: búsqueda semántica')
print('   - calcular_pandas: cálculos estadísticos')
print('   - resumen_dataset: visión general')

✅ Herramientas creadas
   - buscar_rag: búsqueda semántica
   - calcular_pandas: cálculos estadísticos
   - resumen_dataset: visión general


🤖 CELDA 10 - Crear el agente (versión final)

In [14]:
# -*- coding: utf-8 -*-
# CELDA 10 - Crear el agente con LangChain 0.3

from langchain.agents import AgentExecutor
from langchain.memory import ConversationBufferMemory
from langchain.agents import create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage # Ensure these are imported

# Configurar el LLM
llm = ChatMistralAI(
    api_key=MISTRAL_API_KEY,
    model="mistral-small-latest",
    temperature=0,
    max_retries=2
)

# Configurar memoria
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True
)

# Lista de herramientas
tools = [buscar_rag, calcular_pandas, resumen_dataset]

# Definir el prompt para el agente
agent_prompt = ChatPromptTemplate.from_messages([
    SystemMessage(content=f"""Eres un asistente experto en análisis de ventas.\n\nInformación del dataset:\n- {len(df)} filas, {len(df.columns)} columnas\n- Países: {df['country'].nunique()}\n- Años: {sorted(df['year_id'].dropna().unique()) if 'year_id' in df.columns else 'N/A'}\n\nUsa las herramientas disponibles según el tipo de pregunta:\n- buscar_rag: para preguntas específicas (clientes, productos, países)\n- calcular_pandas: para cálculos (sumas, promedios, group by)\n- resumen_dataset: para información general\n\nResponde siempre en español.\n"""),
    MessagesPlaceholder("chat_history"),
    HumanMessage(content="{input}"),
    MessagesPlaceholder("agent_scratchpad"),
])

# Crear el agente (nueva sintaxis de LangChain 0.3)
agent = create_tool_calling_agent(
    llm=llm,
    tools=tools,
    prompt=agent_prompt
)

# Crear el executor
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    memory=memory,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=5
)

print('\n' + '='*60)
print('🎉 AGENTE CREADO EXITOSAMENTE')
print('='*60)
print(f'✅ Versión LangChain: 0.3.x')
print(f'✅ LLM: Mistral Small')
print(f'✅ Herramientas: {len(tools)}')
print(f'✅ Memoria: Activada')
print('='*60)


🎉 AGENTE CREADO EXITOSAMENTE
✅ Versión LangChain: 0.3.x
✅ LLM: Mistral Small
✅ Herramientas: 3
✅ Memoria: Activada


💬 CELDA 11 - Función de consulta

In [15]:
# -*- coding: utf-8 -*-
# CELDA 11 - Función para consultar

def consultar(pregunta, verbose=True):
    """Consulta al agente"""
    try:
        resultado = agent_executor.invoke({"input": pregunta})
        return resultado['output']
    except Exception as e:
        return f"❌ Error: {str(e)}"

def limpiar_memoria():
    """Limpia el historial de conversación"""
    memory.clear()
    print("🔄 Memoria limpiada")

print('✅ Función consultar() lista')

✅ Función consultar() lista


🎮 CELDA 12 - Bot interactivo

In [ ]:
# -*- coding: utf-8 -*-
# CELDA 12 - Bot interactivo

print('\n' + '='*55)
print('   🤖 BOT LANGCHAIN 0.3 + RAG + MISTRAL')
print('='*55)
print('  Preguntame cualquier cosa sobre tus datos:')
print('  • "¿Qué compró el cliente X?"')
print('  • "Suma de ventas por país"')
print('  • "Top 5 productos más vendidos"')
print('  • "Resumen del dataset"')
print('')
print('  Comandos: salir | reset')
print('='*55)
print()

while True:
    try:
        entrada = input('👤 Tú: ').strip()
    except (KeyboardInterrupt, EOFError):
        print('\n👋 ¡Hasta luego!')
        break

    if not entrada:
        continue

    if entrada.lower() == 'salir':
        print('👋 ¡Hasta luego!')
        break

    elif entrada.lower() == 'reset':
        limpiar_memoria()
        continue

    print('  ⏳ Consultando...\n')
    respuesta = consultar(entrada, verbose=True)
    print(f'\n🤖 Bot:\n{respuesta}\n')


   🤖 BOT LANGCHAIN 0.3 + RAG + MISTRAL
  Preguntame cualquier cosa sobre tus datos:
  • "¿Qué compró el cliente X?"
  • "Suma de ventas por país"
  • "Top 5 productos más vendidos"
  • "Resumen del dataset"

  Comandos: salir | reset

👤 Tú: ¿Qué compró el cliente X?
  ⏳ Consultando...



> Entering new AgentExecutor chain...
Por favor, proporciona más detalles o especifica tu pregunta para que pueda ayudarte de manera efectiva. Por ejemplo:

- ¿Necesitas un **resumen general** del dataset?
- ¿Quieres realizar un **cálculo específico** (como sumas, promedios, group by)?
- ¿Buscas información **específica** sobre clientes, productos, países, etc.?

¡Estoy aquí para ayudarte! 😊

> Finished chain.

🤖 Bot:
Por favor, proporciona más detalles o especifica tu pregunta para que pueda ayudarte de manera efectiva. Por ejemplo:

- ¿Necesitas un **resumen general** del dataset?
- ¿Quieres realizar un **cálculo específico** (como sumas, promedios, group by)?
- ¿Buscas información **específica** so

# CELDA 13 - FUNCIÓN DE CONSULTA MEJORADA


In [ ]:
def consultar_rag(pregunta, verbose=True):
    """
    Función principal para consultar al agente.

    Args:
        pregunta (str): Pregunta en lenguaje natural
        verbose (bool): Si True, muestra el proceso interno del agente

    Returns:
        str: Respuesta del agente
    """
    try:
        # Invocar al agente con la pregunta del usuario
        resultado = agent.invoke({"input": pregunta})
        return resultado['output']
    except Exception as e:
        return f"❌ Error al procesar la pregunta: {str(e)}"

def limpiar_memoria():
    """
    Limpia el historial de conversación.
    Útil para empezar una conversación nueva sin contexto previo.
    """
    memory.clear()
    print("🔄 Memoria conversacional limpiada")

def ver_estado():
    """
    Muestra el estado actual del agente y el dataset.
    Útil para debugging.
    """
    print(f"""
    📊 ESTADO ACTUAL:
    - Filas en dataset: {len(df):,}
    - Documentos en RAG: {len(documentos)}
    - Historial conversación: {len(memory.chat_memory.messages) // 2} intercambios
    - Herramientas disponibles: {len(tools)}
    """)

print('✅ Funciones de consulta listas')
print('   - consultar_rag(pregunta) → para preguntar')
print('   - limpiar_memoria() → para reiniciar conversación')
print('   - ver_estado() → para ver estadísticas')


🧪 CELDA 13 - Prueba rápida

In [ ]:
# -*- coding: utf-8 -*-
# CELDA 13 - Probar con una pregunta

pregunta_test = "¿Cuál es el total de ventas y cuántos clientes únicos tenemos?"

print(f"📝 Pregunta: {pregunta_test}\n")
respuesta = consultar(pregunta_test, verbose=True)
print(f"\n🤖 Respuesta:\n{respuesta}")